# Training & Evaluating the model

## 0. Imports & Config

In [1]:
import pandas as pd
import numpy as np
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, confusion_matrix
import joblib

ARTIFACT_PATH = r"C:\Users\User\Documents\Key Future Docs\Projects\Phising URL\artifacts\artifact.joblib"
RANDOM_STATE = 42

COLS_TO_DROP = ['url', 'domain', 'subdomains', 'path', 'tld', 'protocol']

## 1. Load & Prepare the data for training

### A. Load artifact

In [2]:
artifact = joblib.load(ARTIFACT_PATH)
train_df = artifact["train_df"]
test_df = artifact["test_df"]
train_ngrams = artifact["train_ngrams"]
test_ngrams = artifact["test_ngrams"]
vectorizer = artifact["vectorizer"]
vectorizer_cols = artifact["vectorizer_cols"]

print(f"Train df shape: {train_df.shape}, Test df shape: {test_df.shape}")
print(f"Train ngrams shape: {train_ngrams.shape}, Test ngrams shape: {test_ngrams.shape}")
train_df.head()

Train df shape: (657593, 21), Test df shape: (164399, 21)
Train ngrams shape: (657593, 6000), Test ngrams shape: (164399, 6000)


,url,is_phishing,protocol,domain,subdomains,path,tld,is_ip,url_length,domain_length,...,path_length,tld_length,dot_count,domain_dot_count,digit_count,word_count,special_char_count,digit_letter_ratio,hyphen_count,contains_words_count
0,http://reviewrekutan.info/pass/index0.php,True,http,reviewrekutan.info,[reviewrekutan],/pass/index0.php,info,False,41,18,...,16,4,2,1,1,6,7,0.166667,0,0
1,updateinformatiosonlines.blogspot.my,True,http,updateinformatiosonlines.blogspot.my,"[updateinformatiosonlines, blogspot]",,my,False,36,36,...,0,2,2,2,0,3,2,0.000000,0,1
2,montreal-nightclubs.com/events/black-and-blue-...,False,http,montreal-nightclubs.com,[montreal-nightclubs],/events/black-and-blue-2011-olympic-stadium/,com,False,67,23,...,44,3,1,1,4,10,10,0.400000,1,0
3,potbnb.com/dopbcp/alibaba/alibaba/loader1.htm,True,http,potbnb.com,[potbnb],/dopbcp/alibaba/alibaba/loader1.htm,com,False,45,10,...,35,3,2,1,1,7,6,0.142857,0,0
4,https://inffo.hr625.cn/login.php,True,https,inffo.hr625.cn,"[inffo, hr625]",/login.php,cn,False,32,14,...,10,2,3,2,3,6,7,0.500000,0,1


### B. Compress n-grams with Singular Value Decomposition (SVD)

In [3]:
svd = TruncatedSVD(n_components=100, random_state=RANDOM_STATE)
X_ngram_train = svd.fit_transform(train_ngrams)
X_ngram_test = svd.transform(test_ngrams)

### C. Build stack for training

In [4]:
X_train = train_df.drop(columns=COLS_TO_DROP + ['is_phishing']).to_numpy()
X_test = test_df.drop(columns=COLS_TO_DROP + ['is_phishing']).to_numpy()

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [5]:
X_train = np.hstack([X_train, X_ngram_train])
y_train = train_df['is_phishing']

X_test = np.hstack([X_test, X_ngram_test])
y_test = test_df['is_phishing']

## 2. Train an baseline logistic regression model

In [6]:
reg = LogisticRegression(max_iter=1000)
reg.fit(X_train, y_train)

preds = reg.predict(X_test)
probs = reg.predict_proba(X_test)

reg_accuracy = accuracy_score(y_test, preds)
reg_auc = roc_auc_score(y_test, probs[:, 1])
print(f"Accuracy: {reg_accuracy} AUC: {reg_auc}")

Accuracy: 0.9171771117829184 AUC: 0.973868799671371


## 3. Train an MLP

In [7]:
mlp = MLPClassifier(
    hidden_layer_sizes=(128, 64),
    activation='relu',
    solver='adam',
    max_iter=50,
    random_state=RANDOM_STATE,
    early_stopping=True,
)

mlp.fit(X_train, y_train)
print(f"Trained in {mlp.n_iter_} iterations")

Trained in 42 iterations


In [8]:
preds = mlp.predict(X_test)
probs = mlp.predict_proba(X_test) # probs has 2 columns, first is probability of being 0, second is probability of being 1

mlp_accuracy = accuracy_score(y_test, preds)
mlp_auc = roc_auc_score(y_test, probs[:, 1])

print(f"Accuracy: {mlp_accuracy} AUC: {mlp_auc}")

Accuracy: 0.9731750193127695 AUC: 0.9960759073448578


## 4. Evalutate the model

In [9]:
print("Classification Report...")
print(classification_report(y_test, preds))

print("Confusion Matrix...")
print(confusion_matrix(y_test, preds))

Classification Report...
              precision    recall  f1-score   support

       False       0.97      0.98      0.97     85403
        True       0.97      0.97      0.97     78996

    accuracy                           0.97    164399
   macro avg       0.97      0.97      0.97    164399
weighted avg       0.97      0.97      0.97    164399

Confusion Matrix...
[[83405  1998]
 [ 2412 76584]]
